# 1. POD Fundamentals - Theory & Implementation

## Overview
This notebook covers the mathematical foundations of Proper Orthogonal Decomposition (POD) and demonstrates the complete workflow using synthetic lid-driven cavity flow data.

### Learning Objectives
- Understand SVD and POD mode extraction
- Analyze energy content and mode importance
- Reconstruct approximations using reduced basis
- Quantify reconstruction error

## 1. Setup & Data Loading

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import json
from pathlib import Path

# Import POD utilities
from pod_solver import POD_Solver
from data_utils import DataHandler, SnaphotProcessor

print("✓ Imports successful")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Load cavity flow test data
data_dir = Path('../data/cavity_flow')

u_snapshots = DataHandler.load_npy(str(data_dir / 'u_snapshots.npy'))
v_snapshots = DataHandler.load_npy(str(data_dir / 'v_snapshots.npy'))
time_vector = DataHandler.load_npy(str(data_dir / 'time_vector.npy'))

# Load metadata
with open(data_dir / 'parameters.json', 'r') as f:
    metadata = json.load(f)

print(f"\nMetadata:")
for key, val in metadata.items():
    print(f"  {key}: {val}")

In [ ]:
# Combine velocity components into single snapshot matrix
snapshots = np.vstack([u_snapshots, v_snapshots])

print(f"\nSnapshot matrix properties:")
print(f"  Shape (n_spatial, n_snapshots): {snapshots.shape}")
print(f"  Grid resolution: 128 × 128 × 2 = {metadata['grid_size']**2 * 2}")
print(f"  Time range: [{time_vector[0]:.3f}, {time_vector[-1]:.3f}]")
print(f"  Number of modes (max): {min(snapshots.shape)}")

## 2. Data Preprocessing & Visualization

In [ ]:
# Initialize POD solver
pod = POD_Solver(snapshots, verbose=True)

# Preprocess data (mean subtraction)
pod.preprocess()

In [ ]:
# Visualize mean field and sample snapshots
mean_u = pod.mean_field[:len(u_snapshots)].reshape(128, 128)
mean_v = pod.mean_field[len(u_snapshots):].reshape(128, 128)

# Plot sample snapshots
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)

# Row 1: Mean field
ax1 = fig.add_subplot(gs[0, 0])
im1 = ax1.imshow(mean_u, cmap='RdBu_r')
ax1.set_title('Mean u-velocity', fontsize=11, fontweight='bold')
ax1.set_ylabel('y')
plt.colorbar(im1, ax=ax1)

ax2 = fig.add_subplot(gs[0, 1])
im2 = ax2.imshow(mean_v, cmap='RdBu_r')
ax2.set_title('Mean v-velocity', fontsize=11, fontweight='bold')
plt.colorbar(im2, ax=ax2)

# Row 1: Magnitude
ax3 = fig.add_subplot(gs[0, 2])
magnitude = np.sqrt(mean_u**2 + mean_v**2)
im3 = ax3.imshow(magnitude, cmap='viridis')
ax3.set_title('Mean velocity magnitude', fontsize=11, fontweight='bold')
plt.colorbar(im3, ax=ax3)

# Row 2-3: Sample snapshots
snapshot_indices = [0, 125, 250, 375, 500-1]
titles = [f't={time_vector[i]:.2f}' for i in snapshot_indices]

for idx, (snap_idx, title) in enumerate(zip(snapshot_indices[:3], titles[:3])):
    u_snap = snapshots[:len(u_snapshots), snap_idx].reshape(128, 128)
    ax = fig.add_subplot(gs[1, idx])
    im = ax.imshow(u_snap, cmap='RdBu_r')
    ax.set_title(f'u-field {title}', fontsize=10)
    plt.colorbar(im, ax=ax)

for idx, (snap_idx, title) in enumerate(zip(snapshot_indices[2:], titles[2:])):
    v_snap = snapshots[len(u_snapshots):, snap_idx].reshape(128, 128)
    ax = fig.add_subplot(gs[2, idx])
    im = ax.imshow(v_snap, cmap='RdBu_r')
    ax.set_title(f'v-field {title}', fontsize=10)
    plt.colorbar(im, ax=ax)

plt.suptitle('Cavity Flow: Mean Field and Sample Snapshots', fontsize=13, fontweight='bold', y=0.995)
plt.savefig('../results/01_snapshots_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/01_snapshots_visualization.png")

## 3. SVD Computation & POD Mode Extraction

In [ ]:
# Compute SVD using Method of Snapshots
print("Computing SVD...")
pod.compute_svd(method='numpy')

print(f"\nSVD Results:")
print(f"  POD modes shape: {pod.U.shape}")
print(f"  Singular values shape: {pod.s.shape}")
print(f"  Temporal coefficients shape: {pod.Vt.shape}")
print(f"  First 10 singular values: {pod.s[:10]}")

In [ ]:
# Analyze energy content
n_modes_total = len(pod.s)
print(f"\nEnergy Analysis:")
print(f"  Total modes: {n_modes_total}")
print(f"  Energy per mode (first 15):")

for i in range(min(15, len(pod.energy))):
    cum_energy = pod.cumulative_energy[i]
    print(f"    Mode {i+1:2d}: {pod.energy[i]:8.4f} (Cumulative: {cum_energy*100:6.2f}%)")

# Find modes for different energy levels
energy_targets = [90, 95, 99]
print(f"\nModes needed for energy targets:")
for target in energy_targets:
    n_modes = np.argmax(pod.cumulative_energy >= target/100) + 1
    actual_energy = pod.cumulative_energy[n_modes-1] * 100
    print(f"  {target}% energy: {n_modes} modes (actual: {actual_energy:.2f}%)")

## 4. Energy & Error Analysis

In [ ]:
# Plot energy content
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative energy plot
n_plot = 50
ax = axes[0]
ax.semilogy(range(1, n_plot+1), 1 - pod.cumulative_energy[:n_plot], 'b-', linewidth=2, label='1 - Cumulative energy')
ax.axhline(y=0.01, color='r', linestyle='--', label='1% error')
ax.axhline(y=0.05, color='orange', linestyle='--', label='5% error')
ax.set_xlabel('Number of modes', fontsize=11, fontweight='bold')
ax.set_ylabel('Energy deficit', fontsize=11, fontweight='bold')
ax.set_title('Cumulative Energy (log scale)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

# Linear scale
ax = axes[1]
ax.plot(range(1, n_plot+1), pod.cumulative_energy[:n_plot]*100, 'b-', linewidth=2)
ax.axhline(y=90, color='r', linestyle='--', alpha=0.7, label='90%')
ax.axhline(y=95, color='orange', linestyle='--', alpha=0.7, label='95%')
ax.axhline(y=99, color='green', linestyle='--', alpha=0.7, label='99%')
ax.set_xlabel('Number of modes', fontsize=11, fontweight='bold')
ax.set_ylabel('Cumulative energy (%)', fontsize=11, fontweight='bold')
ax.set_title('Energy Content (linear scale)', fontsize=12, fontweight='bold')
ax.set_ylim([50, 100.5])
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig('../results/01_energy_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/01_energy_analysis.png")

In [ ]:
# Compute reconstruction error for different mode counts
n_mode_range = np.array([5, 10, 15, 20, 30, 50, 75, 100])
errors = []

print("Computing reconstruction errors...")
for n_modes in n_mode_range:
    error = pod.error_l2(n_modes, snapshots)
    errors.append(error)
    print(f"  n={n_modes:3d} modes: L2 error = {error*100:.4f}%")

errors = np.array(errors)

In [ ]:
# Plot reconstruction error
fig, ax = plt.subplots(figsize=(10, 6))

ax.loglog(n_mode_range, errors*100, 'bo-', linewidth=2, markersize=8, label='L2 Error')
ax.set_xlabel('Number of modes', fontsize=12, fontweight='bold')
ax.set_ylabel('L2 Reconstruction Error (%)', fontsize=12, fontweight='bold')
ax.set_title('ROM Convergence: Reconstruction Error vs Mode Count', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')

# Add reference lines
ax.axhline(y=1, color='r', linestyle='--', alpha=0.5, label='1% error')
ax.axhline(y=0.1, color='g', linestyle='--', alpha=0.5, label='0.1% error')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../results/01_error_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/01_error_convergence.png")

## 5. POD Mode Visualization

In [ ]:
# Extract first 4 modes and visualize
n_modes_plot = 4
modes = pod.get_modes(n_modes_plot)

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(n_modes_plot, 3, figure=fig, hspace=0.35, wspace=0.3)

for mode_idx in range(n_modes_plot):
    mode = modes[:, mode_idx]
    u_mode = mode[:len(u_snapshots)].reshape(128, 128)
    v_mode = mode[len(u_snapshots):].reshape(128, 128)
    
    # u-component
    ax = fig.add_subplot(gs[mode_idx, 0])
    im = ax.imshow(u_mode, cmap='RdBu_r')
    ax.set_title(f'Mode {mode_idx+1}: u-component\nEnergy: {pod.energy[mode_idx]:.4f}', fontsize=10, fontweight='bold')
    ax.set_ylabel('y')
    plt.colorbar(im, ax=ax)
    
    # v-component
    ax = fig.add_subplot(gs[mode_idx, 1])
    im = ax.imshow(v_mode, cmap='RdBu_r')
    ax.set_title(f'Mode {mode_idx+1}: v-component', fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax)
    
    # Magnitude
    ax = fig.add_subplot(gs[mode_idx, 2])
    magnitude = np.sqrt(u_mode**2 + v_mode**2)
    im = ax.imshow(magnitude, cmap='viridis')
    ax.set_title(f'Mode {mode_idx+1}: Magnitude', fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax)

plt.suptitle('POD Modes (First 4)', fontsize=14, fontweight='bold')
plt.savefig('../results/01_pod_modes.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/01_pod_modes.png")

## 6. Temporal Dynamics & Reconstruction

In [ ]:
# Project snapshots onto POD basis to get temporal coefficients
n_modes_reconstruct = 15
temporal_coeffs = pod.project_onto_modes(snapshots, n_modes_reconstruct)

print(f"Temporal coefficients shape: {temporal_coeffs.shape}")
print(f"  ({n_modes_reconstruct} modes × {temporal_coeffs.shape[1]} time instances)")

In [ ]:
# Plot temporal dynamics of first 4 modes
fig, ax = plt.subplots(figsize=(14, 6))

colors = plt.cm.tab10(np.linspace(0, 1, 4))
for i in range(4):
    ax.plot(time_vector, temporal_coeffs[i, :], color=colors[i], linewidth=2, label=f'Mode {i+1}')

ax.set_xlabel('Time', fontsize=12, fontweight='bold')
ax.set_ylabel('Temporal coefficient', fontsize=12, fontweight='bold')
ax.set_title('POD Temporal Dynamics (First 4 Modes)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='best')

plt.tight_layout()
plt.savefig('../results/01_temporal_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/01_temporal_dynamics.png")

In [ ]:
# Reconstruct snapshots using reduced basis
reconstructed = pod.reconstruct(n_modes_reconstruct, snapshots)

print(f"Reconstruction summary for {n_modes_reconstruct} modes:")
print(f"  Original shape: {snapshots.shape}")
print(f"  Reconstructed shape: {reconstructed.shape}")
print(f"  L2 Error: {pod.error_l2(n_modes_reconstruct, snapshots)*100:.4f}%")
print(f"  Cumulative energy captured: {pod.cumulative_energy[n_modes_reconstruct-1]*100:.4f}%")

In [ ]:
# Compare original and reconstructed snapshots
snap_idx = 250  # Middle snapshot

u_orig = snapshots[:len(u_snapshots), snap_idx].reshape(128, 128)
v_orig = snapshots[len(u_snapshots):, snap_idx].reshape(128, 128)

u_recon = reconstructed[:len(u_snapshots), snap_idx].reshape(128, 128)
v_recon = reconstructed[len(u_snapshots):, snap_idx].reshape(128, 128)

u_error = u_orig - u_recon
v_error = v_orig - v_recon

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)

# Original
ax = fig.add_subplot(gs[0, 0])
im = ax.imshow(u_orig, cmap='RdBu_r')
ax.set_title('Original u', fontsize=11, fontweight='bold')
ax.set_ylabel('y')
plt.colorbar(im, ax=ax)

ax = fig.add_subplot(gs[0, 1])
im = ax.imshow(v_orig, cmap='RdBu_r')
ax.set_title('Original v', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax)

# Reconstructed
ax = fig.add_subplot(gs[1, 0])
im = ax.imshow(u_recon, cmap='RdBu_r')
ax.set_title('Reconstructed u', fontsize=11, fontweight='bold')
ax.set_ylabel('y')
plt.colorbar(im, ax=ax)

ax = fig.add_subplot(gs[1, 1])
im = ax.imshow(v_recon, cmap='RdBu_r')
ax.set_title('Reconstructed v', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax)

# Error
ax = fig.add_subplot(gs[2, 0])
im = ax.imshow(u_error, cmap='seismic')
ax.set_title(f'u Error (max: {np.max(np.abs(u_error)):.2e})', fontsize=11, fontweight='bold')
ax.set_ylabel('y')
ax.set_xlabel('x')
plt.colorbar(im, ax=ax)

ax = fig.add_subplot(gs[2, 1])
im = ax.imshow(v_error, cmap='seismic')
ax.set_title(f'v Error (max: {np.max(np.abs(v_error)):.2e})', fontsize=11, fontweight='bold')
ax.set_xlabel('x')
plt.colorbar(im, ax=ax)

# Magnitude
ax = fig.add_subplot(gs[0, 2])
mag_orig = np.sqrt(u_orig**2 + v_orig**2)
im = ax.imshow(mag_orig, cmap='viridis')
ax.set_title('Original magnitude', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax)

ax = fig.add_subplot(gs[1, 2])
mag_recon = np.sqrt(u_recon**2 + v_recon**2)
im = ax.imshow(mag_recon, cmap='viridis')
ax.set_title('Reconstructed magnitude', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax)

ax = fig.add_subplot(gs[2, 2])
mag_error = mag_orig - mag_recon
im = ax.imshow(mag_error, cmap='seismic')
ax.set_title(f'Magnitude Error', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax)

plt.suptitle(f'Reconstruction Comparison (Snapshot index={snap_idx}, {n_modes_reconstruct} modes)', 
             fontsize=13, fontweight='bold')
plt.savefig('../results/01_reconstruction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to results/01_reconstruction_comparison.png")

## 7. Summary Statistics

In [ ]:
summary = pod.summary()

print("\n" + "="*70)
print("POD ANALYSIS SUMMARY - CAVITY FLOW")
print("="*70)
print(f"\nProblem Setup:")
print(f"  Case: {metadata['case']}")
print(f"  Grid: {metadata['grid_size']} × {metadata['grid_size']} × 2")
print(f"  Snapshots: {n_snapshots}")
print(f"  Spatial DoFs: {summary['n_spatial']}")

print(f"\nPOD Characteristics:")
print(f"  Dominant mode energy: {pod.energy[0]:.6f}")
print(f"  Top 5 modes energy: {np.sum(pod.energy[:5]):.6f}")
print(f"  Top 10 modes energy: {np.sum(pod.energy[:10]):.6f}")

print(f"\nReduced Basis Efficiency:")
energy_targets = [90, 95, 99]
for target in energy_targets:
    n_modes = np.argmax(pod.cumulative_energy >= target/100) + 1
    reduction = 100 * (1 - n_modes / summary['n_spatial'])
    print(f"  {target}% energy: {n_modes:3d} modes (dim reduction: {reduction:5.1f}%)")

print(f"\nReconstruction Quality ({n_modes_reconstruct} modes):")
print(f"  Energy captured: {pod.cumulative_energy[n_modes_reconstruct-1]*100:.4f}%")
print(f"  L2 error: {pod.error_l2(n_modes_reconstruct, snapshots)*100:.6f}%")
print(f"  Dimension reduction: {100*(1-n_modes_reconstruct/summary['n_spatial']):.2f}%")

print(f"\nComputational Complexity:")
print(f"  SVD time: O(n × m²) where n={summary['n_spatial']}, m={n_snapshots}")
print(f"  ROM evaluation: O(n × k) where k={n_modes_reconstruct} << m")
print("\n" + "="*70)

In [ ]:
print("✓ Notebook 1 Complete!")
print("\nKey Takeaways:")
print("  1. POD extracts optimal orthonormal basis from data via SVD")
print("  2. First few modes capture majority of energy (exponential decay)")
print("  3. Dramatic dimensionality reduction possible (16384 -> ~15 DoFs)")
print("  4. Reconstruction error decreases rapidly with mode count")
print("  5. Physical modes reveal dominant flow structures")
print("\nNext: Run Notebook 2 for cavity flow case study analysis")